# SCACM pilot: replacing mean-pooled conditioning in a few-shot CVAE

**Author:** Sadaf Zehra
**Context:** independent project built while studying Stanford CS236 (Deep Generative Models, Stefano Ermon). It is the first stage of my PhD proposal *SCACM: Semantic-Clustered Conditioning for Few-Shot Generation and Zero-Shot Recognition*.

### The question
In my MS thesis I built a few-shot conditional VAE that generates new glyphs of a character from a handful of support images. Like most few-shot generators, it builds its condition by **averaging** the support embeddings. Averaging throws away how the support drawings differ from each other, so samples drift toward a blurry "average" glyph.

This notebook tests a direct replacement for that average and asks one measurable question:

> With the decoder, prior, data and training budget held fixed, does clustered cross-attention conditioning give a better conditional likelihood and more class-specific samples than mean pooling, at 1, 5 and 10 shots, on alphabets the model has never seen?

### What is compared
| Variant | How the support set becomes a condition vector |
|---|---|
| **Mean-pool** (thesis baseline) | average of support embeddings, then an MLP |
| **Attention only** (ablation) | learned queries cross-attend over the individual support embeddings |
| **SCACM** (proposed) | differentiable soft k-means groups the supports into style clusters; queries cross-attend over centroids and instances |

The clustering idea follows SC-CLIP (Quan et al., 2025). The CIZSL++ deviation loss (Elhoseiny et al., 2021) is the planned next stage and is discussed at the end.

### Protocol
* **Data:** Omniglot at 28x28, binarized. Training uses the 30 *background* alphabets (964 characters). All evaluation uses the 20 *evaluation* alphabets (659 characters), so every test character comes from an alphabet never seen in training.
* **Model:** CVAE with a learned conditional prior p(z|c), amortized posterior q(z|x,c) and a Bernoulli decoder p(x|z,c). Trained with the ELBO, reparameterization trick and KL warm-up.
* **Metrics** (held-out targets, never shown as supports):
  * **IWAE NLL** (nats, lower is better): importance-weighted bound on -log p(x|support set), K=50.
  * **Class-fidelity SSIM** (higher is better): SSIM between each generated glyph and the closest held-out drawing of the same character.
  * **20-way retrieval accuracy** (higher is better, chance = 5%): a generated glyph counts as correct if its most similar held-out drawing, among 20 characters, belongs to its own character.
  * **Diversity** (mean pairwise L1 between samples of one character): checks that gains are not bought by collapsing to a single output.
  * A **real-data reference** row puts real held-out drawings through the same SSIM, retrieval and diversity code, which gives a ceiling for the sample metrics.

### How to run
Open in Google Colab, choose *Runtime > Change runtime type > GPU*, then *Runtime > Run all*. Set `FAST_DEV_RUN = True` first if you want a one-minute check that the whole pipeline runs end to end.

In [ ]:
import os, math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import pandas as pd

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = "outputs"
os.makedirs(OUT, exist_ok=True)
os.makedirs("data", exist_ok=True)
print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | device: {DEVICE}")
if DEVICE.type == "cpu":
    print("No GPU found. The notebook still works, but full training will be slow. In Colab: Runtime > Change runtime type > GPU.")

## 1. Configuration
All three variants share every setting below. Only the conditioning module changes.

In [ ]:
FAST_DEV_RUN = False   # True = tiny run (about a minute) to check the pipeline end to end

CFG = dict(
    img_size=28, bin_threshold=0.3,
    emb_dim=128, cond_dim=128, z_dim=32,
    n_clusters=3, kmeans_iters=3, n_heads=4, n_queries=2,
    batch_classes=32, n_targets=4, max_shot=10,
    steps=8000, lr=1e-3, kl_warmup_frac=0.3, grad_clip=5.0,
    eval_shots=(1, 5, 10), eval_way=20, n_gen=4, iwae_k=50,
    seeds=(0,),            # use (0, 1, 2) for mean and spread over three runs
)
EVAL_MAX_CLASSES = None    # None = all 659 evaluation characters

if FAST_DEV_RUN:
    CFG.update(steps=30, iwae_k=5)
    EVAL_MAX_CLASSES = 40

# supports (up to max_shot) + held-out targets + real reference drawings must fit in 20 drawings per character
assert CFG["max_shot"] + CFG["n_targets"] + CFG["n_gen"] <= 20
assert CFG["n_gen"] >= 2
CFG

## 2. Data
Each Omniglot character has 20 drawings by different people. Images are resized to 28x28, inverted (strokes = 1) and binarized so that the Bernoulli likelihood, and therefore the NLL, is well defined. The tensors are cached after the first load.

In [ ]:
def load_omniglot(background):
    tf = transforms.Compose([transforms.Resize((CFG["img_size"], CFG["img_size"])), transforms.ToTensor()])
    ds = torchvision.datasets.Omniglot(root="data", background=background, download=True, transform=tf)
    per_class = {}
    for i in range(len(ds)):
        x, y = ds[i]
        x = ((1.0 - x) > CFG["bin_threshold"]).float()   # strokes -> 1, paper -> 0
        per_class.setdefault(y, []).append(x)
    classes = sorted(per_class)
    return torch.stack([torch.stack(per_class[c][:20]) for c in classes])   # [C, 20, 1, 28, 28]

cache = "data/omniglot28_bin.pt"
if os.path.exists(cache):
    X_train, X_test = torch.load(cache)
else:
    print("Loading Omniglot (first run takes a few minutes)...")
    X_train, X_test = load_omniglot(True), load_omniglot(False)
    torch.save((X_train, X_test), cache)

X_train, X_test = X_train.to(DEVICE), X_test.to(DEVICE)
print("train (background alphabets):", tuple(X_train.shape))
print("test  (evaluation alphabets):", tuple(X_test.shape))
print(f"stroke pixel fraction: {X_train.mean().item():.3f}")

fig, ax = plt.subplots(1, 1, figsize=(10, 3))
ax.imshow(make_grid(X_test[:6, :10].reshape(-1, 1, 28, 28).cpu(), nrow=10, padding=2, pad_value=0.5)[0], cmap="gray_r")
ax.set_title("6 test characters, 10 drawings each (unseen alphabets)")
ax.axis("off")
plt.show()

In [ ]:
def sample_episode(X, n_classes, k_shot, n_targets):
    # k_shot supports and n_targets disjoint targets for each of n_classes random characters
    C, n_draw = X.shape[:2]
    cls = torch.randperm(C)[:n_classes].to(X.device)
    perm = torch.argsort(torch.rand(n_classes, n_draw), dim=1).to(X.device)
    S = X[cls[:, None], perm[:, :k_shot]]                    # [B, k, 1, 28, 28]
    T = X[cls[:, None], perm[:, k_shot:k_shot + n_targets]]  # [B, t, 1, 28, 28]
    return S, T

S, T = sample_episode(X_train, 4, 5, 3)
print("support:", tuple(S.shape), "| targets:", tuple(T.shape))

## 3. Model

### 3.1 Conditioning modules
Each module maps support embeddings `H` with shape `[B, k, d]` to a condition vector `c` with shape `[B, cond_dim]`, so the rest of the CVAE is identical across variants.

**SCACM** runs a few iterations of differentiable soft k-means. Centroids start at the support mean plus learned offsets, so the module is permutation invariant and still works with a single shot. Learned queries then attend over the centroids and the individual supports, with a type embedding marking which is which. The mean is kept as a residual input so the module can fall back to it when that is the best choice.

Squared distances are computed directly (no `cdist`), because the gradient of a Euclidean distance at zero is undefined and a one-shot cluster sits exactly on its support point.

In [ ]:
class MeanPoolCond(nn.Module):
    def __init__(self, d, cond_dim):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, cond_dim), nn.SiLU(), nn.Linear(cond_dim, cond_dim))
        self.last_attn, self.last_assign = None, None

    def forward(self, H):
        return self.net(H.mean(1))


def soft_kmeans(H, init, iters, tau, eps=1e-3):
    # H: [B, k, d], init: [B, M, d] -> centroids [B, M, d], soft assignments [B, k, M]
    mu = init
    a = None
    for _ in range(iters):
        d2 = (H.unsqueeze(2) - mu.unsqueeze(1)).pow(2).sum(-1)       # [B, k, M]
        a = torch.softmax(-d2 / tau, dim=-1)
        mass = a.sum(1).unsqueeze(-1)                                 # [B, M, 1]
        mu = (a.transpose(1, 2) @ H + eps * mu) / (mass + eps)        # eps keeps empty clusters stable
    return mu, a


class AttnCond(nn.Module):
    # use_clusters=False -> "Attention only" ablation; use_clusters=True -> SCACM
    def __init__(self, d, cond_dim, n_heads, n_queries, use_clusters, n_clusters=3, iters=3):
        super().__init__()
        self.use_clusters, self.iters = use_clusters, iters
        self.norm = nn.LayerNorm(d)
        self.queries = nn.Parameter(torch.randn(n_queries, d) * 0.02)
        self.attn = nn.MultiheadAttention(d, n_heads, batch_first=True)
        if use_clusters:
            self.offsets = nn.Parameter(torch.randn(n_clusters, d) * 0.5)
            self.log_tau = nn.Parameter(torch.tensor(math.log(d)))
            self.type_emb = nn.Parameter(torch.zeros(2, d))           # 0 = centroid token, 1 = instance token
        self.net = nn.Sequential(nn.Linear(d * (n_queries + 1), cond_dim), nn.SiLU(), nn.Linear(cond_dim, cond_dim))
        self.last_attn, self.last_assign = None, None

    def forward(self, H):
        B = H.shape[0]
        H = self.norm(H)
        mean = H.mean(1)
        tokens = H
        if self.use_clusters:
            init = mean.unsqueeze(1) + self.offsets.unsqueeze(0)
            mu, assign = soft_kmeans(H, init, self.iters, self.log_tau.exp())
            tokens = torch.cat([mu + self.type_emb[0], H + self.type_emb[1]], dim=1)
            self.last_assign = assign.detach()
        q = self.queries.unsqueeze(0).expand(B, -1, -1)
        out, w = self.attn(q, tokens, tokens, need_weights=True)      # w: [B, n_queries, n_tokens]
        self.last_attn = w.detach()
        return self.net(torch.cat([out.reshape(B, -1), mean], dim=-1))

### 3.2 Few-shot CVAE
* support encoder: CNN, one embedding per support image
* learned conditional prior p(z | c)
* amortized posterior q(z | x, c)
* Bernoulli decoder p(x | z, c)

Training minimizes the negative ELBO, `E_q[-log p(x|z,c)] + beta * KL(q(z|x,c) || p(z|c))`, with beta warmed up linearly from 0 to 1 to reduce posterior collapse.

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, 2, 1), nn.GroupNorm(8, cout), nn.SiLU())

class ConvEncoder(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        self.net = nn.Sequential(conv_block(1, 32), conv_block(32, 64), conv_block(64, 128),   # 28 -> 14 -> 7 -> 4
                                 nn.Flatten(), nn.Linear(128 * 4 * 4, out_dim))
    def forward(self, x):
        return self.net(x)

def gaussian_kl(mu_q, lv_q, mu_p, lv_p):
    return 0.5 * (lv_p - lv_q + (lv_q.exp() + (mu_q - mu_p).pow(2)) / lv_p.exp() - 1.0).sum(-1)

def log_normal(z, mu, lv):
    return -0.5 * (math.log(2 * math.pi) + lv + (z - mu).pow(2) / lv.exp()).sum(-1)


class FewShotCVAE(nn.Module):
    def __init__(self, conditioner, emb_dim, cond_dim, z_dim):
        super().__init__()
        self.support_enc = ConvEncoder(emb_dim)
        self.cond = conditioner
        self.x_enc = ConvEncoder(256)
        self.q_net = nn.Sequential(nn.Linear(256 + cond_dim, 256), nn.SiLU(), nn.Linear(256, 2 * z_dim))
        self.p_net = nn.Sequential(nn.Linear(cond_dim, 256), nn.SiLU(), nn.Linear(256, 2 * z_dim))
        self.dec_fc = nn.Sequential(nn.Linear(z_dim + cond_dim, 128 * 7 * 7), nn.SiLU())
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.GroupNorm(8, 64), nn.SiLU(),   # 7 -> 14
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.GroupNorm(8, 32), nn.SiLU(),    # 14 -> 28
            nn.Conv2d(32, 1, 3, 1, 1))

    def condition(self, S):
        B, k = S.shape[:2]
        H = self.support_enc(S.reshape(B * k, *S.shape[2:])).reshape(B, k, -1)
        return self.cond(H)

    def prior(self, c):
        mu, lv = self.p_net(c).chunk(2, dim=-1)
        return mu, lv.clamp(-8, 8)

    def posterior(self, x, c):
        mu, lv = self.q_net(torch.cat([self.x_enc(x), c], dim=-1)).chunk(2, dim=-1)
        return mu, lv.clamp(-8, 8)

    def decode(self, z, c):
        return self.dec(self.dec_fc(torch.cat([z, c], dim=-1)).reshape(-1, 128, 7, 7))   # logits

    def forward(self, S, T, beta=1.0):
        B, t = T.shape[:2]
        c = self.condition(S).repeat_interleave(t, dim=0)
        x = T.reshape(B * t, *T.shape[2:])
        mu_q, lv_q = self.posterior(x, c)
        mu_p, lv_p = self.prior(c)
        z = mu_q + torch.randn_like(mu_q) * (0.5 * lv_q).exp()                 # reparameterization trick
        rec = F.binary_cross_entropy_with_logits(self.decode(z, c), x, reduction="none").flatten(1).sum(1)
        kl = gaussian_kl(mu_q, lv_q, mu_p, lv_p)
        return (rec + beta * kl).mean(), rec.mean().detach(), kl.mean().detach()

    @torch.no_grad()
    def sample(self, S, n):
        # n new glyphs per character, z drawn from the learned conditional prior; returns Bernoulli means
        c = self.condition(S).repeat_interleave(n, dim=0)
        mu, lv = self.prior(c)
        z = mu + torch.randn_like(mu) * (0.5 * lv).exp()
        return torch.sigmoid(self.decode(z, c)).reshape(S.shape[0], n, *S.shape[2:])


VARIANTS = {"Mean-pool (baseline)": "mean", "Attention only": "attn", "SCACM": "scacm"}

def build_model(variant, cfg):
    d, cd = cfg["emb_dim"], cfg["cond_dim"]
    if variant == "mean":
        cond = MeanPoolCond(d, cd)
    elif variant == "attn":
        cond = AttnCond(d, cd, cfg["n_heads"], cfg["n_queries"], use_clusters=False)
    elif variant == "scacm":
        cond = AttnCond(d, cd, cfg["n_heads"], cfg["n_queries"], use_clusters=True,
                        n_clusters=cfg["n_clusters"], iters=cfg["kmeans_iters"])
    else:
        raise ValueError(variant)
    return FewShotCVAE(cond, d, cd, cfg["z_dim"])

for name, v in VARIANTS.items():
    m = build_model(v, CFG)
    n_all = sum(p.numel() for p in m.parameters())
    n_cond = sum(p.numel() for p in m.cond.parameters())
    print(f"{name:22s} total params {n_all/1e6:.2f}M | conditioning module {n_cond/1e3:.1f}K")

# shape check on one batch for every variant (catches wiring errors before the long run)
S, T = sample_episode(X_train, 2, 3, 2)
for name, v in VARIANTS.items():
    m = build_model(v, CFG).to(DEVICE)
    loss, rec, kl = m(S, T)
    assert torch.isfinite(loss), name
    assert m.sample(S, 2).shape == (2, 2, 1, 28, 28), name
print("forward and sampling check passed for all variants")

## 4. Evaluation code
* **IWAE bound** (Burda et al., 2016): `log p(x|S) >= E[log (1/K) sum_k p(x, z_k | S) / q(z_k | x, S)]`. Tighter than the ELBO and a standard way to compare likelihood models.
* **SSIM** is implemented in PyTorch (7x7 uniform window, the same defaults as `skimage.metrics.structural_similarity`) so that all pairwise comparisons run as one batched operation.
* The evaluation episodes use a fixed seed, so every model sees exactly the same characters, supports and targets. Supports are nested (the 1-shot support is part of the 5-shot support, which is part of the 10-shot support) and the targets are the same at every shot count.

In [ ]:
@torch.no_grad()
def iwae_nll(model, S, T, K):
    B, t = T.shape[:2]
    c = model.condition(S).repeat_interleave(t, dim=0)
    x = T.reshape(B * t, *T.shape[2:])
    mu_q, lv_q = model.posterior(x, c)
    mu_p, lv_p = model.prior(c)
    log_w = []
    for _ in range(K):
        z = mu_q + torch.randn_like(mu_q) * (0.5 * lv_q).exp()
        log_px = -F.binary_cross_entropy_with_logits(model.decode(z, c), x, reduction="none").flatten(1).sum(1)
        log_w.append(log_px + log_normal(z, mu_p, lv_p) - log_normal(z, mu_q, lv_q))
    return -(torch.logsumexp(torch.stack(log_w), dim=0) - math.log(K))      # [B*t], nats


def ssim_batch(a, b, win=7, data_range=1.0):
    # a, b: [P, 1, H, W] -> mean SSIM per pair, [P]
    C1, C2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
    pool = lambda x: F.avg_pool2d(x, win, stride=1)
    n = win * win
    cov_norm = n / (n - 1)
    mu_a, mu_b = pool(a), pool(b)
    var_a = cov_norm * (pool(a * a) - mu_a ** 2)
    var_b = cov_norm * (pool(b * b) - mu_b ** 2)
    cov_ab = cov_norm * (pool(a * b) - mu_a * mu_b)
    s = ((2 * mu_a * mu_b + C1) * (2 * cov_ab + C2)) / ((mu_a ** 2 + mu_b ** 2 + C1) * (var_a + var_b + C2))
    return s.flatten(1).mean(1)


def fidelity_metrics(gen, T):
    # gen: [way, G, 1, H, W] candidate glyphs, T: [way, nT, 1, H, W] held-out real drawings
    way, G = gen.shape[:2]
    nT = T.shape[1]
    hw = gen.shape[2:]
    A = gen.reshape(way * G, *hw)
    Bt = T.reshape(way * nT, *hw)
    Pa = A[:, None].expand(-1, way * nT, *hw).reshape(-1, *hw)
    Pb = Bt[None].expand(way * G, -1, *hw).reshape(-1, *hw)
    sim = ssim_batch(Pa, Pb).reshape(way * G, way, nT).max(-1).values      # best match within each character
    own = torch.arange(way, device=gen.device).repeat_interleave(G)
    fid = sim[torch.arange(way * G, device=gen.device), own]
    acc = (sim.argmax(1) == own).float()
    pair_l1 = (gen[:, :, None] - gen[:, None]).abs().flatten(3).mean(-1)   # [way, G, G]
    div = pair_l1.sum((1, 2)) / (G * (G - 1))
    return fid, acc, div


def eval_episodes(X, cfg, seed, max_classes):
    g = torch.Generator().manual_seed(seed)
    n = X.shape[0] if max_classes is None else min(max_classes, X.shape[0])
    order = torch.randperm(X.shape[0], generator=g)[:n]
    perm = torch.argsort(torch.rand(n, X.shape[1], generator=g), dim=1)
    way = cfg["eval_way"]
    for s in range(0, n - way + 1, way):
        yield order[s:s + way].to(X.device), perm[s:s + way].to(X.device)


def summarize(vals):
    out = {}
    for key, chunks in vals.items():
        v = torch.cat(chunks).float()
        out[key] = v.mean().item()
        out[key + "_ci95"] = 1.96 * v.std().item() / math.sqrt(len(v)) if len(v) > 1 else float("nan")
    return out


@torch.no_grad()
def evaluate(model, X, k_shot, cfg, seed=123, max_classes=None):
    model.eval()
    torch.manual_seed(seed + k_shot)
    ms, nT, G = cfg["max_shot"], cfg["n_targets"], cfg["n_gen"]
    vals = {"nll": [], "ssim": [], "acc": [], "div": []}
    for cls, p in eval_episodes(X, cfg, seed, max_classes):
        S = X[cls[:, None], p[:, :k_shot]]
        T = X[cls[:, None], p[:, ms:ms + nT]]
        vals["nll"].append(iwae_nll(model, S, T, cfg["iwae_k"]).cpu())
        fid, acc, div = fidelity_metrics(model.sample(S, G), T)
        vals["ssim"].append(fid.cpu()); vals["acc"].append(acc.cpu()); vals["div"].append(div.cpu())
    return summarize(vals)


@torch.no_grad()
def real_data_reference(X, cfg, seed=123, max_classes=None):
    # real held-out drawings scored with the same code: a ceiling for the sample metrics
    ms, nT, G = cfg["max_shot"], cfg["n_targets"], cfg["n_gen"]
    vals = {"ssim": [], "acc": [], "div": []}
    for cls, p in eval_episodes(X, cfg, seed, max_classes):
        T = X[cls[:, None], p[:, ms:ms + nT]]
        R = X[cls[:, None], p[:, ms + nT:ms + nT + G]]
        fid, acc, div = fidelity_metrics(R, T)
        vals["ssim"].append(fid.cpu()); vals["acc"].append(acc.cpu()); vals["div"].append(div.cpu())
    return summarize(vals)

# sanity checks for the metric code
x = X_test[:1, :1].reshape(1, 1, 28, 28)
assert abs(ssim_batch(x, x).item() - 1.0) < 1e-4, "SSIM of an image with itself must be 1"
ref = real_data_reference(X_test, CFG, max_classes=EVAL_MAX_CLASSES)
print("real-data reference:", {k: round(v, 4) for k, v in ref.items()})

## 5. Training
Every model gets the same seed, step count, optimizer and episode sampler. The shot count is drawn uniformly from 1 to 10 at each step, so a single model serves all shot settings.

In [ ]:
def train_model(variant, seed, cfg):
    set_seed(seed)
    model = build_model(variant, cfg).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["steps"])
    warm = max(1, int(cfg["kl_warmup_frac"] * cfg["steps"]))
    log_every = max(1, cfg["steps"] // 50)
    hist, t0 = [], time.time()
    model.train()
    for step in range(1, cfg["steps"] + 1):
        k = random.randint(1, cfg["max_shot"])
        S, T = sample_episode(X_train, cfg["batch_classes"], k, cfg["n_targets"])
        beta = min(1.0, step / warm)
        loss, rec, kl = model(S, T, beta)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
        opt.step()
        sched.step()
        if step == 1 or step % log_every == 0:
            hist.append(dict(step=step, neg_elbo=(rec + kl).item(), rec=rec.item(), kl=kl.item(), beta=beta))
        if step % max(1, cfg["steps"] // 5) == 0:
            print(f"  step {step:5d} | -ELBO {(rec + kl).item():7.2f} | rec {rec.item():7.2f} | KL {kl.item():6.2f} | {time.time() - t0:5.0f}s")
    return model, pd.DataFrame(hist), time.time() - t0


results, curves, models = [], {}, {}
for seed in CFG["seeds"]:
    for name, variant in VARIANTS.items():
        print(f"\n=== {name} | seed {seed} ===")
        model, hist, secs = train_model(variant, seed, CFG)
        curves[(name, seed)] = hist
        models[(name, seed)] = model
        torch.save(model.state_dict(), f"{OUT}/{variant}_seed{seed}.pt")
        for k in CFG["eval_shots"]:
            r = evaluate(model, X_test, k, CFG, max_classes=EVAL_MAX_CLASSES)
            r.update(model=name, seed=seed, shots=k, train_sec=secs)
            results.append(r)
            print(f"  eval {k:2d}-shot | NLL {r['nll']:.2f} | SSIM {r['ssim']:.4f} | 20-way acc {r['acc']:.3f} | div {r['div']:.4f}")

## 6. Results

In [ ]:
df = pd.DataFrame(results)
df.to_csv(f"{OUT}/results.csv", index=False)

metrics = ["nll", "ssim", "acc", "div"]
summary = df.groupby(["model", "shots"], sort=False)[metrics].agg(["mean", "std"]) if len(CFG["seeds"]) > 1 \
    else df.set_index(["model", "shots"])[metrics]
display(summary.round(4))

print("Real-data reference (ceiling for the sample metrics):")
print(f"  SSIM {ref['ssim']:.4f} | 20-way acc {ref['acc']:.3f} | div {ref['div']:.4f}")

nll_table = df.pivot_table(index="model", columns="shots", values="nll", aggfunc="mean").reindex(list(VARIANTS))
print("\nIWAE NLL in nats (lower is better):")
display(nll_table.round(2))

In [ ]:
first_seed = CFG["seeds"][0]
fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))

for name in VARIANTS:
    h = curves[(name, first_seed)]
    axes[0].plot(h["step"], h["neg_elbo"].rolling(5, min_periods=1).mean(), label=name)
axes[0].set_title("Training -ELBO (nats, smoothed)"); axes[0].set_xlabel("step"); axes[0].legend(fontsize=8)

titles = {"nll": "Test IWAE NLL (lower is better)", "ssim": "Class-fidelity SSIM (higher is better)",
          "acc": "20-way retrieval acc (higher is better)"}
for ax, key in zip(axes[1:], ["nll", "ssim", "acc"]):
    for name in VARIANTS:
        sub = df[df.model == name].groupby("shots")[key].mean()
        ax.plot(sub.index, sub.values, marker="o", label=name)
    if key in ("ssim", "acc"):
        ax.axhline(ref[key], ls="--", c="gray", lw=1, label="real data")
    ax.set_title(titles[key]); ax.set_xlabel("shots"); ax.set_xticks(list(CFG["eval_shots"]))
axes[3].legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT}/results.png", dpi=150, bbox_inches="tight")
plt.show()

### Qualitative comparison
Five supports from an unseen alphabet, four samples from each model (drawn from the learned prior), and four real held-out drawings of the same character.

In [ ]:
def grid_row(imgs):
    return make_grid(imgs.cpu(), nrow=imgs.shape[0], padding=2, pad_value=0.5)[0]

n_show, k_show = 6, 5
g = torch.Generator().manual_seed(2024)
cls = torch.randperm(X_test.shape[0], generator=g)[:n_show].to(DEVICE)
S_show = X_test[cls, :k_show]
R_show = X_test[cls, 16:20]

columns = [("Support (5-shot)", S_show)]
for name in VARIANTS:
    torch.manual_seed(0)
    models[(name, first_seed)].eval()
    columns.append((name, models[(name, first_seed)].sample(S_show, 4)))
columns.append(("Real held-out", R_show))

fig, axes = plt.subplots(n_show, len(columns), figsize=(3.3 * len(columns), 1.1 * n_show))
for r in range(n_show):
    for c, (title, imgs) in enumerate(columns):
        axes[r, c].imshow(grid_row(imgs[r]), cmap="gray_r", vmin=0, vmax=1)
        axes[r, c].axis("off")
        if r == 0:
            axes[r, c].set_title(title, fontsize=10)
plt.tight_layout()
plt.savefig(f"{OUT}/samples.png", dpi=150, bbox_inches="tight")
plt.show()

### What SCACM attends to
For one 10-shot test character: the cluster each support drawing is assigned to, and how much attention the learned queries give to each centroid and each individual drawing.

In [ ]:
scacm = models[("SCACM", first_seed)].eval()
g = torch.Generator().manual_seed(7)
c_id = torch.randperm(X_test.shape[0], generator=g)[:1].to(DEVICE)
S_one = X_test[c_id, :10]                                    # [1, 10, 1, 28, 28]
with torch.no_grad():
    _ = scacm.condition(S_one)
assign = scacm.cond.last_assign[0].argmax(-1).cpu()          # cluster id of each support
attn = scacm.cond.last_attn[0].mean(0).cpu()                 # average over queries -> [M + k]
M = CFG["n_clusters"]

fig = plt.figure(figsize=(14, 4.6))
for i in range(10):
    ax = fig.add_subplot(2, 10, i + 1)
    ax.imshow(S_one[0, i, 0].cpu(), cmap="gray_r")
    ax.set_title(f"cluster {assign[i].item()}", fontsize=8)
    ax.axis("off")
axb = fig.add_subplot(2, 1, 2)
labels = [f"C{m}" for m in range(M)] + [f"s{i}" for i in range(10)]
colors = ["tab:orange"] * M + ["tab:blue"] * 10
axb.bar(labels, attn.numpy(), color=colors)
axb.set_ylabel("attention weight")
axb.set_title("Query attention over centroids (orange) and individual supports (blue)")
plt.tight_layout()
plt.savefig(f"{OUT}/scacm_attention.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Plain-language summary generated from the numbers above
base, prop = "Mean-pool (baseline)", "SCACM"
lines = []
for k in CFG["eval_shots"]:
    b = df[(df.model == base) & (df.shots == k)][metrics].mean()
    s = df[(df.model == prop) & (df.shots == k)][metrics].mean()
    lines.append(f"{k:2d}-shot: NLL {b.nll:.2f} -> {s.nll:.2f} ({s.nll - b.nll:+.2f} nats), "
                 f"SSIM {b.ssim:.4f} -> {s.ssim:.4f}, 20-way acc {b.acc:.3f} -> {s.acc:.3f}, "
                 f"diversity {b['div']:.4f} -> {s['div']:.4f}")
print(f"{base}  ->  {prop}")
print("\n".join(lines))
if FAST_DEV_RUN:
    print("\nFAST_DEV_RUN is on: these numbers come from a 30-step run and say nothing about the methods.")

## 7. Reading the results honestly
* **NLL is the primary metric.** It scores how well the condition explains real held-out drawings, so a model cannot improve it by producing one sharp but generic glyph. Differences inside the 95% intervals (stored as `*_ci95` in `outputs/results.csv`) should not be called improvements.
* **Look at SSIM and retrieval accuracy together with diversity.** Higher fidelity with much lower diversity points to mode collapse, not better conditioning.
* **The ablation separates the two ingredients.** If "Attention only" already matches SCACM, the gain comes from attending to individual supports and the clustering step adds little. If SCACM wins mainly at 5 and 10 shots, that fits the hypothesis, since clusters only carry information once there are several drawings to group.
* **One shot is a control.** With a single support there is nothing to cluster, so SCACM should roughly match the baseline there.
* **Run three seeds** (`seeds=(0, 1, 2)`) before stating any conclusion.
* These numbers are not comparable to my thesis results (SSIM 0.7822, FID 12.21), which used a different resolution, preprocessing, split and metric definitions.

## 8. Concepts from CS236 used here
| CS236 topic | Where it appears |
|---|---|
| Latent variable models, ELBO | CVAE objective in `FewShotCVAE.forward` |
| Amortized inference, reparameterization trick | `posterior` network and `z = mu + eps * sigma` |
| Learned (conditional) priors | `p_net` gives p(z \| support set) instead of N(0, I) |
| Posterior collapse, KL annealing | linear beta warm-up |
| Likelihood evaluation, importance sampling | IWAE bound in `iwae_nll` |
| Evaluating generative models beyond likelihood | SSIM fidelity, retrieval accuracy, diversity, real-data ceiling |

## 9. Next steps (PhD proposal stages)
1. **Deviation loss from CIZSL++** (Elhoseiny et al., 2021): add an entropy-based term that controls how far samples for an unseen character move from the seen classes, and measure its effect on diversity and NLL.
2. **Perceptual or adversarial feedback:** a lightweight discriminator head on top of the CVAE decoder, compared at matched NLL.
3. **Harder writing systems:** multi-stroke fonts and Urdu handwriting, where intra-class style variation is much larger than in Omniglot and averaging should hurt most.
4. **Zero-shot recognition:** reuse the same conditioning module over CLIP features of video frames (UCF-101, HMDB-51) to test whether clustered conditioning transfers beyond generation.

### References
* Kingma and Welling. Auto-Encoding Variational Bayes. ICLR 2014.
* Sohn, Lee and Yan. Learning Structured Output Representation using Deep Conditional Generative Models. NeurIPS 2015.
* Burda, Grosse and Salakhutdinov. Importance Weighted Autoencoders. ICLR 2016.
* Lake, Salakhutdinov and Tenenbaum. Human-level concept learning through probabilistic program induction. Science 2015.
* Elhoseiny, Yi and Elfeki. CIZSL++: Creativity Inspired Generative Zero-Shot Learning. 2021.
* Quan et al. SC-CLIP. 2025.
* Stanford CS236: Deep Generative Models, https://deepgenerativemodels.github.io